# LLM Reasoning Pipeline

Lightweight Colab workflow for uncertainty-focused Hugging Face reasoning with `google/flan-t5-small`.

In [ ]:
!pip install transformers accelerate -q

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import numpy as np

path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed/'

X = np.load(path + 'X_test.npy')[:20]
y = np.load(path + 'y_test.npy')[:20]

In [ ]:
X = X.reshape(X.shape[0], -1)
X.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X, y)

preds = model.predict(X)
probs = model.predict_proba(X)

In [ ]:
import os
import sys

repo_candidates = ['/content/AI_Agentic_DL', '/content/your-repo-name']
repo_path = next((path for path in repo_candidates if os.path.exists(path)), None)
if repo_path is None:
    raise FileNotFoundError('Place the local repo in /content/AI_Agentic_DL or update repo_path.')

os.chdir(repo_path)
sys.path.append(repo_path)

from llm_reasoning.llm_pipeline import LLMPipeline

print('Using repo:', repo_path)

In [ ]:
indices = [i for i in range(len(preds)) if max(probs[i]) < 0.7][:10]

if len(indices) == 0:
    indices = list(range(min(10, len(preds))))

indices

In [ ]:
pipeline = LLMPipeline(model='google/flan-t5-small')
# Optional upgrade: LLMPipeline(model='google/flan-t5-base')

results = []

for i in indices:
    sample = X[i]
    pred = preds[i]
    conf = max(probs[i])

    prompt, output = pipeline.run(sample, pred, conf)

    results.append({
        'index': int(i),
        'prediction': int(pred),
        'confidence': float(conf),
        'reasoning': output
    })

    print(f'\nSample {i}')
    print(output)

In [ ]:
import json

save_path = os.path.join(repo_path, 'experiments', 'results')
os.makedirs(save_path, exist_ok=True)

with open(os.path.join(save_path, 'llm_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved successfully')
print('Saved to:', os.path.join(save_path, 'llm_results.json'))